# Credit Risk Scoring & Expected Loss Analysis
## Notebook 1: Data Loading & Preprocessing

This notebook loads the raw *Default of Credit Card Clients* dataset from the UCI Machine Learning Repository, renames the cryptic `X1–X23` column codes to meaningful field names, remaps undocumented categorical codes, and saves the cleaned dataset to `data/processed/` for all subsequent notebooks.

**Dataset**: [Default of Credit Card Clients — UCI ML Repository (ID 350)](https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients)  
**Instances**: 30,000 credit card clients (Taiwan, April–September 2005)  
**Features**: 23 predictors + 1 binary target (`DEFAULT_NEXT_MONTH`)  

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import os

RAW_PATH  = '../data/raw/default_of_credit_card_clients.csv'
SAVE_PATH = '../data/processed/cleaned_credit_card_data.csv'

## 2. Load Raw Data

The raw CSV uses coded column names (`X1` through `X23`, `Y`). We rename them to the descriptive field names from the original paper (Yeh & Lien, 2009).

In [2]:
raw_df = pd.read_csv(RAW_PATH)
print(f'Raw dataset shape: {raw_df.shape}')
print(f'Columns: {raw_df.columns.tolist()}')

Raw dataset shape: (30000, 24)
Columns: ['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8', 'X9', 'X10', 'X11', 'X12', 'X13', 'X14', 'X15', 'X16', 'X17', 'X18', 'X19', 'X20', 'X21', 'X22', 'X23', 'Y']


## 3. Rename Columns

| Original | Renamed | Description |
|----------|---------|-------------|
| X1 | LIMIT_BAL | Approved credit limit (NTD) |
| X2 | SEX | 1=male, 2=female |
| X3 | EDUCATION | 1=graduate school, 2=university, 3=high school, 4=other |
| X4 | MARRIAGE | 1=married, 2=single, 3=other |
| X5 | AGE | Age in years |
| X6–X11 | PAY_0, PAY_2–PAY_6 | Repayment status (Sep–Apr 2005) |
| X12–X17 | BILL_AMT1–6 | Bill statement amounts (NTD) |
| X18–X23 | PAY_AMT1–6 | Amounts paid (NTD) |
| Y | DEFAULT_NEXT_MONTH | 1 = defaulted, 0 = did not default |

In [3]:
column_rename = {
    'X1':  'LIMIT_BAL',  'X2':  'SEX',      'X3':  'EDUCATION',
    'X4':  'MARRIAGE',   'X5':  'AGE',
    'X6':  'PAY_0',      'X7':  'PAY_2',     'X8':  'PAY_3',
    'X9':  'PAY_4',      'X10': 'PAY_5',     'X11': 'PAY_6',
    'X12': 'BILL_AMT1',  'X13': 'BILL_AMT2', 'X14': 'BILL_AMT3',
    'X15': 'BILL_AMT4',  'X16': 'BILL_AMT5', 'X17': 'BILL_AMT6',
    'X18': 'PAY_AMT1',   'X19': 'PAY_AMT2',  'X20': 'PAY_AMT3',
    'X21': 'PAY_AMT4',   'X22': 'PAY_AMT5',  'X23': 'PAY_AMT6',
    'Y':   'DEFAULT_NEXT_MONTH'
}
df = raw_df.rename(columns=column_rename)
print(f'Renamed columns: {df.columns.tolist()}')

Renamed columns: ['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'DEFAULT_NEXT_MONTH']


## 4. Remap Undocumented Categorical Codes

The dataset contains undocumented values in `EDUCATION` (codes 0, 5, 6) and `MARRIAGE` (code 0). These are consolidated into the "Other" category following standard practice for this dataset.

- **EDUCATION**: 0, 5, 6 → 4 (Other)
- **MARRIAGE**: 0 → 3 (Other)

In [4]:
# EDUCATION: 1=graduate school, 2=university, 3=high school, 4=other
df['EDUCATION'] = df['EDUCATION'].replace({0: 4, 5: 4, 6: 4})

# MARRIAGE: 1=married, 2=single, 3=other
df['MARRIAGE'] = df['MARRIAGE'].replace({0: 3})

print('EDUCATION value counts after remapping:')
print(df['EDUCATION'].value_counts().sort_index())
print()
print('MARRIAGE value counts after remapping:')
print(df['MARRIAGE'].value_counts().sort_index())

EDUCATION value counts after remapping:
EDUCATION
1    10585
2    14030
3     4917
4      468
Name: count, dtype: int64

MARRIAGE value counts after remapping:
MARRIAGE
1    13659
2    15964
3      377
Name: count, dtype: int64


## 5. Data Quality Check

In [5]:
print(f'Rows    : {len(df):,}')
print(f'Columns : {len(df.columns)}')
print()
print('Missing values:')
missing = df.isnull().sum()
if missing.sum() == 0:
    print('  None — dataset is complete.')
else:
    print(missing[missing > 0])
print()
print('Data types:')
print(df.dtypes)

Rows    : 30,000
Columns : 24

Missing values:
  None — dataset is complete.

Data types:
LIMIT_BAL             int64
SEX                   int64
EDUCATION             int64
MARRIAGE              int64
AGE                   int64
PAY_0                 int64
PAY_2                 int64
PAY_3                 int64
PAY_4                 int64
PAY_5                 int64
PAY_6                 int64
BILL_AMT1             int64
BILL_AMT2             int64
BILL_AMT3             int64
BILL_AMT4             int64
BILL_AMT5             int64
BILL_AMT6             int64
PAY_AMT1              int64
PAY_AMT2              int64
PAY_AMT3              int64
PAY_AMT4              int64
PAY_AMT5              int64
PAY_AMT6              int64
DEFAULT_NEXT_MONTH    int64
dtype: object


## 6. Save Cleaned Dataset

The cleaned dataframe is saved to `data/processed/` so all subsequent notebooks can load from a consistent, validated source.

In [6]:
os.makedirs('../data/processed', exist_ok=True)
df.to_csv(SAVE_PATH, index=False)
print(f'Saved: {SAVE_PATH}')
print(f'Shape: {df.shape}')
df.head(3)

Saved: ../data/processed/cleaned_credit_card_data.csv
Shape: (30000, 24)


,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,DEFAULT_NEXT_MONTH
0,20000,2,2,1,24,2,2,-1,-1,-2,...,0,0,0,0,689,0,0,0,0,1
1,120000,2,2,2,26,-1,2,0,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,90000,2,2,2,34,0,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
